# Preprocessing DisasterWatch

Pipeline preprocessing lengkap data bencana BNPB (2015–2026) untuk menghasilkan dataset siap ML.

**Output yang dihasilkan:**
| File | Deskripsi |
|------|-----------|
| `bnpb_disaster_clean.csv` | Dataset bersih tanpa feature engineering |
| `bnpb_disaster_features.csv` | Dataset lengkap dengan semua fitur turunan |
| `bnpb_disaster_cluster_ready.csv` | Fitur siap clustering (provinsi + kabkot level) |
| `bnpb_disaster_clf_ready.csv` | Fitur siap klasifikasi severity |
| `preprocessing_report.txt` | Laporan ringkasan preprocessing |

## 0. Setup & Load Raw Data

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler, RobustScaler
from sklearn.impute import SimpleImputer

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 100)

ROOT  = Path.cwd().parent if Path.cwd().name == 'notebook' else Path.cwd()
OUT   = ROOT / 'data' / 'processed'


# ── Sumber data tunggal ───────────────────────────────────────────────────────
# Notebook ini dan notebook EDA WAJIB memakai file yang sama. Jangan memilih
# dengan sorted()[-1] (urutan alfabetis) atau ukuran file: keduanya pernah
# menghasilkan pilihan berbeda (2026 saja vs 2015-2026).
# Aturan: ambil CSV bnpb_bencana_*.csv dengan rentang tanggal terluas.
def pick_raw_csv(folder):
    candidates = list(Path(folder).glob('bnpb_bencana_*.csv'))
    if not candidates:
        raise FileNotFoundError(f'Tidak ada bnpb_bencana_*.csv di {folder}')

    def coverage(path):
        try:
            start, end = path.stem.split('_')[-2:]
            return (pd.Timestamp(end) - pd.Timestamp(start)).days, pd.Timestamp(end)
        except (ValueError, IndexError):
            return (-1, pd.Timestamp.min)

    return max(candidates, key=coverage)


INPUT = pick_raw_csv(OUT)

print(f'Input : {INPUT.name}')
print(f'Output: {OUT}')

df_raw = pd.read_csv(INPUT, encoding='utf-8-sig', low_memory=False)
print(f'Shape awal: {df_raw.shape}')
print(f'Kolom  : {df_raw.columns.tolist()}')
df_raw.head(3)

## 1. Audit Kualitas Data Awal

In [ ]:
# ── Tipe data, missing, kardinalitas ─────────────────────────────────────────
audit = pd.DataFrame({
    'dtype':       df_raw.dtypes,
    'non_null':    df_raw.notna().sum(),
    'null_count':  df_raw.isna().sum(),
    'null_pct':    (df_raw.isna().mean() * 100).round(2),
    'unique':      df_raw.nunique(),
    'sample':      [df_raw[c].dropna().iloc[0] if df_raw[c].notna().any() else None
                    for c in df_raw.columns],
})
display(audit)

print(f'\nTotal baris : {len(df_raw):,}')
print(f'Total kolom : {df_raw.shape[1]}')
print(f'Kolom semua null: {list(audit[audit.non_null == 0].index)}')
print(f'Kolom >80% null : {list(audit[audit.null_pct > 80].index)}')

In [ ]:
# ── Visualisasi missing value ─────────────────────────────────────────────────
miss_nonzero = audit[audit.null_pct > 0].sort_values('null_pct', ascending=True)

fig, ax = plt.subplots(figsize=(10, max(4, len(miss_nonzero) * 0.3)))
bars = ax.barh(miss_nonzero.index, miss_nonzero.null_pct,
               color=['#d62728' if v > 80 else '#ff7f0e' if v > 50 else '#1f77b4'
                      for v in miss_nonzero.null_pct])
ax.axvline(50, color='orange', ls='--', lw=1, label='50% threshold')
ax.axvline(80, color='red',    ls='--', lw=1, label='80% threshold')
ax.set(title='Missing Value per Kolom (%)', xlabel='Missing %', xlim=(0, 105))
ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# ── Cek duplikasi ─────────────────────────────────────────────────────────────
dup_full = df_raw.duplicated().sum()
key_cols = ['tahun', 'bulan', 'tanggal', 'id_kab', 'kejadian', 'lokasi']
dup_key  = df_raw.duplicated(subset=[c for c in key_cols if c in df_raw.columns], keep=False).sum()

print(f'Duplikat baris penuh   : {dup_full:,}')
print(f'Duplikat kunci bisnis  : {dup_key:,}')

# Distribusi nilai unik kolom kategorikal kunci
for col in ['kejadian', 'provinsi', 'kabkot']:
    if col in df_raw.columns:
        vc = df_raw[col].value_counts()
        print(f'\n{col}: {len(vc)} nilai unik, top-5:')
        print(vc.head(5).to_string())

## 2. Pembersihan Data (Cleaning)

In [ ]:
# ── Step 1: Salin raw ─────────────────────────────────────────────────────────
df = df_raw.copy()
report_log = []  # log perubahan

# ── Step 2: Kolom numerik ─────────────────────────────────────────────────────
numeric_cols = [
    'id_prov', 'id_kab', 'id_jenis_bencana', 'tahun', 'bulan', 'tanggal',
    'meninggal', 'terluka', 'hilang', 'menderita', 'mengungsi',
    'jumlah_kejadian', 'terendam',
    'rumah_rusak', 'rumah_rusak_berat', 'rumah_rusak_sedang', 'rumah_rusak_ringan',
    'lahan', 'sawah', 'kebun', 'hutan', 'kolam', 'irigasi',
    'jalan', 'jembatan_rusak', 'kerugian_juta',
    'pendidikan_rusak', 'peribadatan_rusak', 'kesehatan_rusak',
    'kantor_rusak', 'fasum_rusak', 'kios_rusak', 'pabrik_rusak',
    'longitude', 'latitude', 'geometry_lon', 'geometry_lat',
    'lng_prov', 'lat_prov',
]
converted = 0
for col in numeric_cols:
    if col in df.columns:
        before = df[col].dtype
        df[col] = pd.to_numeric(df[col], errors='coerce')
        if str(before) != str(df[col].dtype):
            converted += 1
report_log.append(f'Konversi numerik: {converted} kolom')

# ── Step 3: String/kategori ───────────────────────────────────────────────────
str_cols = ['provinsi', 'kabkot', 'desa', 'lokasi', 'alamat_lengkap',
            'kejadian', 'nama_kejadian', 'penyebab', 'kronologis', 'sumber']
for col in str_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()
        df[col] = df[col].replace({'nan': np.nan, 'None': np.nan, '': np.nan})

# Standardisasi kejadian → UPPER
if 'kejadian' in df.columns:
    df['kejadian'] = df['kejadian'].str.upper().str.strip()
if 'provinsi' in df.columns:
    df['provinsi'] = df['provinsi'].str.upper().str.strip()

report_log.append('Standarisasi string: strip + upper pada kolom kunci')
print('\n'.join(report_log))
display(df[['provinsi', 'kabkot', 'kejadian']].head(5))

In [ ]:
# ── Step 4: Parsing tanggal ───────────────────────────────────────────────────
n_before = len(df)
df['date'] = pd.to_datetime(
    dict(year=df.tahun, month=df.bulan, day=df.tanggal), errors='coerce')

invalid_date = df['date'].isna().sum()
print(f'Tanggal tidak valid: {invalid_date:,} baris')

# Hapus baris tanpa tanggal valid
df = df.dropna(subset=['date'])
report_log.append(f'Drop baris tanggal invalid: {n_before - len(df)}')

print(f'Rentang tanggal: {df.date.min().date()} → {df.date.max().date()}')
print(f'Shape setelah cleaning tanggal: {df.shape}')

In [ ]:
# ── Step 5: Drop baris tanpa identitas wajib ──────────────────────────────────
mandatory_cols = ['kejadian', 'provinsi', 'kabkot']
mandatory_cols = [c for c in mandatory_cols if c in df.columns]
n_before = len(df)
df = df.dropna(subset=mandatory_cols)
report_log.append(f'Drop tanpa identitas wajib: {n_before - len(df)}')
print(f'Drop tanpa identitas wajib: {n_before - len(df):,} → shape: {df.shape}')

# ── Step 6: Hapus duplikat penuh ─────────────────────────────────────────────
n_before = len(df)
df = df.drop_duplicates()
report_log.append(f'Drop duplikat penuh: {n_before - len(df)}')
print(f'Drop duplikat penuh: {n_before - len(df):,} → shape: {df.shape}')

In [ ]:
# ── Step 7: Pembersihan nilai extrem / anomali koordinat ──────────────────────
# Koordinat valid Indonesia: lon 90–145, lat -12 s.d. 8
for lon_col in ['geometry_lon', 'longitude']:
    if lon_col in df.columns:
        invalid = ~df[lon_col].between(90, 145)
        df.loc[invalid, lon_col] = np.nan

for lat_col in ['geometry_lat', 'latitude']:
    if lat_col in df.columns:
        invalid = ~df[lat_col].between(-12, 8)
        df.loc[invalid, lat_col] = np.nan

# Clip nilai dampak negatif → 0
impact_cols_clip = [
    'meninggal', 'terluka', 'hilang', 'menderita', 'mengungsi',
    'rumah_rusak', 'terendam', 'lahan', 'sawah', 'kebun', 'hutan',
    'jembatan_rusak', 'jalan', 'kerugian_juta', 'jumlah_kejadian'
]
for col in impact_cols_clip:
    if col in df.columns:
        n_neg = (df[col] < 0).sum()
        if n_neg > 0:
            print(f'  Clip nilai negatif {col}: {n_neg} baris')
        df[col] = df[col].clip(lower=0)

report_log.append('Koordinat di luar batas Indonesia di-null-kan')
report_log.append('Nilai negatif kolom dampak di-clip ke 0')
print(f'Shape setelah cleaning anomali: {df.shape}')

In [ ]:
# ── Step 8: Penanganan missing value per grup kolom ───────────────────────────
# Prinsip: dampak yang tidak dilaporkan TETAP NaN, tidak diimputasi 0.
# Mengisi 0 mengubah "tidak diketahui" menjadi "tidak ada dampak" dan
# mendistorsi label severity. Ketidakhadiran data dicatat lewat flag.
impact_cols = [
    'meninggal', 'terluka', 'hilang', 'menderita', 'mengungsi',
    'rumah_rusak', 'rumah_rusak_berat', 'rumah_rusak_sedang', 'rumah_rusak_ringan',
    'terendam', 'lahan', 'sawah', 'kebun', 'hutan',
    'jembatan_rusak', 'jalan', 'jumlah_kejadian',
    'pendidikan_rusak', 'peribadatan_rusak', 'kesehatan_rusak',
    'kantor_rusak', 'fasum_rusak', 'kios_rusak', 'pabrik_rusak',
    'kerugian_juta',
]
impact_cols = [c for c in impact_cols if c in df.columns]

# Flag per kolom: 1 = tidak dilaporkan (NaN), 0 = ada nilai
for col in impact_cols:
    df[f'flag_{col}_missing'] = df[col].isna().astype(int)

# Kolom inti yang menentukan label severity
core_impact = ['meninggal', 'hilang', 'terluka', 'menderita', 'mengungsi',
               'rumah_rusak', 'terendam']
core_impact = [c for c in core_impact if c in df.columns]

# Ringkasan kelengkapan dampak per baris
df['impact_missing_count']      = df[impact_cols].isna().sum(axis=1)
df['impact_report_ratio']       = 1 - df['impact_missing_count'] / len(impact_cols)
df['core_impact_missing_count'] = df[core_impact].isna().sum(axis=1)
df['has_impact_report']         = (df['core_impact_missing_count'] < len(core_impact)).astype(int)
df['has_full_impact_report']    = (df['core_impact_missing_count'] == 0).astype(int)

# Koordinat & teks: tetap NaN (tidak diimputasi)
df['has_coordinates'] = (
    df['geometry_lon'].notna() & df['geometry_lat'].notna()
).astype(int)

df['has_kronologis'] = df.get('kronologis', pd.Series(dtype=str)).notna().astype(int)

report_log.append(f'Kolom dampak ({len(impact_cols)}) tetap NaN + flag missing')
report_log.append('Ringkasan kelengkapan dampak per baris dibuat')

print('Missing per kolom dampak inti (%):')
display(df[core_impact].isna().mean().mul(100).round(1).to_frame('missing_pct').T)
print(f"Baris tanpa laporan dampak sama sekali: {(df.has_impact_report == 0).sum():,}")
print(f"Baris dengan laporan dampak lengkap   : {(df.has_full_impact_report == 1).sum():,}")

print('\nMissing setelah penanganan (dampak tetap NaN):')
miss_after = df.isna().sum()
display(miss_after[miss_after > 0].to_frame('null_count').assign(
    null_pct=(df.isna().mean()*100).round(2)))

## 3. Simpan Dataset Bersih

In [ ]:
clean_path = OUT / 'bnpb_disaster_clean.csv'
df.to_csv(clean_path, index=False, encoding='utf-8-sig')
print(f'✓ Dataset bersih disimpan: {clean_path}')
print(f'  Shape: {df.shape}')
display(df.head(3))

## 4. Feature Engineering Temporal

In [ ]:
# ── Fitur waktu dasar ─────────────────────────────────────────────────────────
df['year']          = df.date.dt.year
df['month']         = df.date.dt.month
df['quarter']       = df.date.dt.quarter
df['dayofweek']     = df.date.dt.dayofweek         # 0=Senin
df['dayofyear']     = df.date.dt.dayofyear
df['week_of_year']  = df.date.dt.isocalendar().week.astype(int)
df['year_month']    = df.date.dt.to_period('M').astype(str)
df['year_quarter']  = df.date.dt.to_period('Q').astype(str)

# ── Konteks musiman Indonesia ─────────────────────────────────────────────────
# Musim hujan (1=ya, 0=tidak): Nov–Mar dominan hujan di sebagian besar Indonesia
df['musim_hujan']   = df.month.isin([11, 12, 1, 2, 3]).astype(int)

# Fase musim (lebih granular)
def musim_fase(m):
    if m in [12, 1, 2]:    return 'puncak_hujan'
    elif m in [3, 4]:      return 'transisi_kemarau'
    elif m in [5, 6, 7, 8, 9]: return 'kemarau'
    else:                  return 'transisi_hujan'  # 10, 11

df['musim_fase']    = df.month.map(musim_fase)

# Encoding cyclical untuk month (sin/cos agar model ML tahu bulan 12 dekat bulan 1)
df['month_sin']     = np.sin(2 * np.pi * df.month / 12)
df['month_cos']     = np.cos(2 * np.pi * df.month / 12)
df['dow_sin']       = np.sin(2 * np.pi * df.dayofweek / 7)
df['dow_cos']       = np.cos(2 * np.pi * df.dayofweek / 7)

print('Fitur temporal berhasil dibuat:')
temporal_cols = ['year', 'month', 'quarter', 'dayofweek', 'week_of_year',
                 'musim_hujan', 'musim_fase', 'month_sin', 'month_cos']
display(df[temporal_cols].head(5))

## 5. Feature Engineering Dampak & Severity

In [ ]:
# ── Fitur dampak agregat (NaN-aware) ─────────────────────────────────────────
# Jumlahkan hanya komponen yang dilaporkan. Jika SEMUA komponen NaN → NaN
# ("tidak diketahui"), bukan 0.
def sum_impact(*cols):
    cols = [c for c in cols if c in df.columns]
    if not cols:
        return pd.Series(np.nan, index=df.index)
    sub = df[cols]
    return sub.sum(axis=1, skipna=True).where(sub.notna().any(axis=1))

df['korban_jiwa']        = sum_impact('meninggal', 'hilang')
df['korban_total']       = sum_impact('meninggal', 'hilang', 'terluka')
df['people_affected']    = sum_impact('menderita', 'mengungsi')
df['human_impact_total'] = sum_impact('meninggal', 'hilang', 'terluka',
                                      'menderita', 'mengungsi')
df['material_damage']    = sum_impact('rumah_rusak', 'terendam')
df['agri_damage']        = sum_impact('sawah', 'kebun', 'lahan')
df['infra_damage']       = sum_impact('jembatan_rusak', 'jalan')
df['fasilitas_damage']   = sum_impact('pendidikan_rusak', 'peribadatan_rusak',
                                      'kesehatan_rusak', 'kantor_rusak', 'fasum_rusak')

# ── Flag dampak: 1 = ada dampak, 0 = dilaporkan nol, NaN = tidak dilaporkan ──
def positive_flag(series):
    return pd.Series(
        np.where(series > 0, 1, np.where(series.notna(), 0, np.nan)),
        index=series.index,
    )

df['ada_korban_jiwa']   = positive_flag(df['korban_jiwa'])
df['ada_korban_total']  = positive_flag(df['korban_total'])
df['ada_pengungsi']     = positive_flag(sum_impact('mengungsi'))
df['ada_menderita']     = positive_flag(sum_impact('menderita'))
df['ada_material']      = positive_flag(df['material_damage'])
df['ada_agri']          = positive_flag(df['agri_damage'])
df['ada_infra']         = positive_flag(df['infra_damage'])

# ── Log-transform (NaN tetap NaN) ─────────────────────────────────────────────
log_cols = ['korban_jiwa', 'korban_total', 'people_affected', 'material_damage',
            'agri_damage', 'infra_damage', 'kerugian_juta', 'human_impact_total']
for col in log_cols:
    if col in df.columns:
        df[f'log_{col}'] = np.log1p(df[col])

# ── Severity label (4 kelas + 'Tidak Diketahui') ──────────────────────────────
# Threshold domain knowledge BNPB:
#   Berat  : ≥5 korban jiwa ATAU ≥500 terdampak ATAU ≥100 unit rusak
#   Sedang : ≥1 korban jiwa ATAU ≥50 terdampak  ATAU ≥10 unit rusak
#   Ringan : dampak terukur > 0, semua komponen penentu diketahui
#   Tanpa  : semua komponen penentu diketahui dan bernilai 0
# Baris dengan komponen penentu NaN tidak dipaksa menjadi 'Tanpa Dampak':
# diberi label 'Tidak Diketahui' dan severity_code = NaN (tidak untuk training).
def severity_label(row):
    kj, kt = row['korban_jiwa'], row['korban_total']
    pa, md = row['people_affected'], row['material_damage']
    if (kj >= 5) or (pa >= 500) or (md >= 100):
        return 'Berat'
    if (kj >= 1) or (pa >= 50) or (md >= 10):
        return 'Sedang'
    if pd.isna(kj) or pd.isna(kt) or pd.isna(pa) or pd.isna(md):
        return 'Tidak Diketahui'
    if (kt > 0) or (pa > 0) or (md > 0):
        return 'Ringan'
    return 'Tanpa Dampak'


df['severity']      = df.apply(severity_label, axis=1)
df['severity_code'] = df['severity'].map(
    {'Tanpa Dampak': 0, 'Ringan': 1, 'Sedang': 2, 'Berat': 3})

# Persentase severity hanya dari baris yang labelnya diketahui
def pct_severity(series, level):
    known = series[series.isin(['Tanpa Dampak', 'Ringan', 'Sedang', 'Berat'])]
    return (known == level).mean() * 100 if len(known) else np.nan

# ── Severity numerik alternatif (composite score 0–100) ───────────────────────
# Normalisasi masing-masing komponen ke 0–1 lalu bobot. Komponen NaN tidak
# dihitung; skor menjadi NaN hanya jika semua komponen tidak dilaporkan.
def safe_norm(series, cap_quantile=0.99):
    cap = series.quantile(cap_quantile)
    if not np.isfinite(cap) or cap <= 0:
        return pd.Series(np.where(series.notna(), 0.0, np.nan), index=series.index)
    return (series.clip(upper=cap) / (cap + 1e-9)).clip(0, 1)

score_components = [
    (df['korban_jiwa'], 0.30),
    (df['people_affected'], 0.25),
    (df['material_damage'], 0.20),
    (df.get('kerugian_juta', pd.Series(np.nan, index=df.index)), 0.15),
    (df['agri_damage'], 0.10),
]
has_any_component = pd.concat(
    [s.notna() for s, _ in score_components], axis=1).any(axis=1)
df['severity_score'] = (
    sum(safe_norm(s).fillna(0) * w for s, w in score_components) * 100
).where(has_any_component)

severity_order  = ['Berat', 'Sedang', 'Ringan', 'Tanpa Dampak', 'Tidak Diketahui']
severity_colors = ['#d62728', '#ff7f0e', '#2ca02c', '#aec7e8', '#6C7A89']
order = severity_order[:4]  # kelas yang dipakai dataset klasifikasi

print('Distribusi severity:')
display(df['severity'].value_counts().reindex(severity_order)
        .to_frame('count')
        .assign(pct=lambda d: (d['count'] / len(df) * 100).round(2)))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['severity'].value_counts().reindex(severity_order).plot(
    kind='bar', ax=axes[0], color=severity_colors, title='Distribusi Severity')
axes[0].set_xlabel('')
df['severity_score'].hist(bins=50, ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set(title='Distribusi Severity Score (0–100)', xlabel='Score', ylabel='Frekuensi')
plt.tight_layout(); plt.show()

## 6. Feature Engineering Spasial & Kontekstual

In [ ]:
# ── Koordinat valid ───────────────────────────────────────────────────────────
geo_lon = 'geometry_lon' if 'geometry_lon' in df.columns else 'longitude'
geo_lat = 'geometry_lat' if 'geometry_lat' in df.columns else 'latitude'

df['lon'] = pd.to_numeric(df.get(geo_lon), errors='coerce')
df['lat'] = pd.to_numeric(df.get(geo_lat), errors='coerce')

# Filter batas Indonesia
df.loc[~df.lon.between(90, 145), 'lon'] = np.nan
df.loc[~df.lat.between(-12, 8),  'lat'] = np.nan

# ── Region Indonesia (heuristik berdasarkan id_prov) ─────────────────────────
# Kode provinsi BNPB: 1x=Sumatera, 3x=Jawa, 6x=Kalimantan, 7x=Sulawesi, 8x=NTT/NTB, 9x=Papua
def region_from_prov(prov_id):
    try:
        p = int(prov_id)
        if   10 <= p <= 19: return 'Sumatera'
        elif 31 <= p <= 36: return 'Jawa'
        elif p == 51:       return 'Bali'
        elif p in [52, 53]: return 'Nusa_Tenggara'
        elif 61 <= p <= 65: return 'Kalimantan'
        elif 71 <= p <= 76: return 'Sulawesi'
        elif p in [81, 82]: return 'Maluku'
        elif p in [91, 92, 94, 95, 96, 97]: return 'Papua'
        else:               return 'Lainnya'
    except:
        return 'Unknown'

if 'id_prov' in df.columns:
    df['region'] = df['id_prov'].apply(region_from_prov)

# ── Statistik historis per kabkot (rolling window kontekstual) ───────────────
# Frekuensi historis per kabkot sebagai fitur risiko
kabkot_freq = df.groupby('kabkot')['date'].count().rename('kabkot_total_laporan')
df = df.merge(kabkot_freq.reset_index(), on='kabkot', how='left')

# Frekuensi historis per kabkot × jenis bencana
kabkot_type_freq = (df.groupby(['kabkot', 'kejadian']).size()
                    .reset_index(name='kabkot_type_freq'))
df = df.merge(kabkot_type_freq, on=['kabkot', 'kejadian'], how='left')

print('Fitur spasial & kontekstual:')
display(df[['provinsi', 'kabkot', 'lon', 'lat', 'region', 
            'kabkot_total_laporan', 'kabkot_type_freq']].head(5))

## 7. Encoding Variabel Kategorikal

In [ ]:
# ── Label Encoding ────────────────────────────────────────────────────────────
le_dict = {}
cat_encode_cols = ['kejadian', 'provinsi', 'kabkot', 'musim_fase']
if 'region' in df.columns:
    cat_encode_cols.append('region')

for col in cat_encode_cols:
    if col in df.columns:
        le = LabelEncoder()
        df[f'{col}_code'] = le.fit_transform(df[col].fillna('Unknown').astype(str))
        le_dict[col] = le
        print(f'  {col}: {len(le.classes_)} kelas → {col}_code')

# ── Target Encoding (mean severity_score per kategori) ──────────────────────
for col in ['kejadian', 'provinsi']:
    if col in df.columns:
        mean_enc = df.groupby(col)['severity_score'].mean().rename(f'{col}_mean_severity')
        df = df.merge(mean_enc.reset_index(), on=col, how='left')
        print(f'  Target encoding {col}_mean_severity ✓')

# ── One-Hot Encoding musim_fase (sedikit kelas) ───────────────────────────────
if 'musim_fase' in df.columns:
    ohe_musim = pd.get_dummies(df['musim_fase'], prefix='musim')
    df = pd.concat([df, ohe_musim], axis=1)
    print(f'  OHE musim_fase: {list(ohe_musim.columns)}')

display(df[[c for c in df.columns if c.endswith('_code') or c.endswith('_mean_severity')]].head(3))

## 8. Normalisasi / Scaling

In [ ]:
# ── Fitur numerik untuk scaling ───────────────────────────────────────────────
scale_cols = [
    'korban_jiwa', 'korban_total', 'people_affected', 'material_damage',
    'agri_damage', 'infra_damage', 'kerugian_juta',
    'kabkot_total_laporan', 'kabkot_type_freq',
    'kejadian_mean_severity', 'provinsi_mean_severity'
]
scale_cols = [c for c in scale_cols if c in df.columns]

# RobustScaler & MinMaxScaler sklearn mendukung NaN: NaN diabaikan saat fit dan
# tetap NaN saat transform, sehingga "tidak dilaporkan" tidak berubah jadi 0.
scaler_robust = RobustScaler()
X_robust = scaler_robust.fit_transform(df[scale_cols])
df_robust = pd.DataFrame(X_robust, columns=[f'robust_{c}' for c in scale_cols], index=df.index)

scaler_minmax = MinMaxScaler()
X_minmax = scaler_minmax.fit_transform(df[scale_cols])
df_minmax = pd.DataFrame(X_minmax, columns=[f'mm_{c}' for c in scale_cols], index=df.index)

# Gabungkan
df = pd.concat([df, df_robust, df_minmax], axis=1)

print(f'Fitur setelah scaling ditambahkan: {len(df.columns)} kolom total')
print(f"Baris tanpa nilai korban jiwa (tetap NaN): {df['korban_jiwa'].isna().sum():,}")
display(df[[f'robust_{c}' for c in scale_cols[:4]]].describe().round(3))

## 9. Dataset Siap Clustering

In [ ]:
# ── Agregasi per kabupaten/kota → fitur untuk clustering ─────────────────────
# sum(min_count=1) menjaga NaN jika seluruh nilai dalam grup tidak dilaporkan;
# mean otomatis mengabaikan NaN.
cluster_agg = df.groupby(['id_kab', 'provinsi', 'kabkot']).agg(
    total_laporan=('date', 'size'),
    total_korban_jiwa=('korban_jiwa', lambda x: x.sum(min_count=1)),
    total_korban_total=('korban_total', lambda x: x.sum(min_count=1)),
    total_terdampak=('people_affected', lambda x: x.sum(min_count=1)),
    total_material=('material_damage', lambda x: x.sum(min_count=1)),
    total_agri=('agri_damage', lambda x: x.sum(min_count=1)),
    total_infra=('infra_damage', lambda x: x.sum(min_count=1)),
    total_kerugian=('kerugian_juta', lambda x: x.sum(min_count=1)),
    jenis_bencana_unik=('kejadian', 'nunique'),
    avg_severity_score=('severity_score', 'mean'),
    pct_severity_berat=('severity', lambda x: pct_severity(x, 'Berat')),
    pct_severity_sedang=('severity', lambda x: pct_severity(x, 'Sedang')),
    pct_musim_hujan=('musim_hujan', 'mean'),
    avg_korban_per_event=('korban_total', 'mean'),
    avg_terdampak_per_event=('people_affected', 'mean'),
    std_korban=('korban_total', 'std'),
    n_tanpa_label=('severity', lambda x: (x == 'Tidak Diketahui').sum()),
).reset_index()

# Normalisasi untuk clustering (NaN hanya diimputasi pada input scaler)
cluster_feat_cols = [
    'total_laporan', 'total_korban_jiwa', 'total_terdampak', 'total_material',
    'total_agri', 'total_kerugian', 'jenis_bencana_unik', 'avg_severity_score',
    'pct_severity_berat', 'avg_korban_per_event', 'avg_terdampak_per_event'
]
X_cluster = cluster_agg[cluster_feat_cols].fillna(0)
scaler_cluster = RobustScaler()
X_cluster_scaled = scaler_cluster.fit_transform(X_cluster)
df_cluster_scaled = pd.DataFrame(X_cluster_scaled, columns=cluster_feat_cols)

# Gabung metadata
cluster_ready = pd.concat([cluster_agg[['id_kab', 'provinsi', 'kabkot']],
                            cluster_agg[cluster_feat_cols],
                            df_cluster_scaled.add_prefix('scaled_')], axis=1)

cluster_path = OUT / 'bnpb_disaster_cluster_ready.csv'
cluster_ready.to_csv(cluster_path, index=False, encoding='utf-8-sig')
print(f'✓ Dataset clustering disimpan: {cluster_path}')
print(f'  Shape: {cluster_ready.shape}')
display(cluster_ready.head(5))

In [ ]:
# ── Agregasi per provinsi → untuk clustering tingkat provinsi ─────────────────
prov_cluster = df.groupby(['id_prov', 'provinsi']).agg(
    total_laporan=('date', 'size'),
    total_korban=('korban_total', lambda x: x.sum(min_count=1)),
    total_terdampak=('people_affected', lambda x: x.sum(min_count=1)),
    total_material=('material_damage', lambda x: x.sum(min_count=1)),
    total_kerugian=('kerugian_juta', lambda x: x.sum(min_count=1)),
    jenis_unik=('kejadian', 'nunique'),
    kabkot_unik=('kabkot', 'nunique'),
    avg_severity=('severity_score', 'mean'),
    pct_berat=('severity', lambda x: pct_severity(x, 'Berat')),
    pct_musim_hujan=('musim_hujan', 'mean'),
    n_tanpa_label=('severity', lambda x: (x == 'Tidak Diketahui').sum()),
).reset_index()

prov_path = OUT / 'bnpb_province_cluster_ready.csv'
prov_cluster.to_csv(prov_path, index=False, encoding='utf-8-sig')
print(f'✓ Cluster provinsi disimpan: {prov_path}  |  shape: {prov_cluster.shape}')

## 10. Dataset Siap Klasifikasi

In [ ]:
# ── Fitur untuk klasifikasi severity ─────────────────────────────────────────
# Skenario A: Post-event (semua fitur dampak tersedia)
clf_post_features = [
    # Kontekstual
    'kejadian_code', 'provinsi_code', 'kabkot_code',
    'year', 'month', 'quarter', 'musim_hujan',
    'month_sin', 'month_cos',
    # Target encoded
    'kejadian_mean_severity', 'provinsi_mean_severity',
    # Historis
    'kabkot_total_laporan', 'kabkot_type_freq',
    # Dampak (NaN = tidak dilaporkan; imputasi dilakukan di pipeline model)
    'meninggal', 'hilang', 'terluka', 'menderita', 'mengungsi',
    'rumah_rusak', 'terendam', 'lahan', 'kerugian_juta',
    'jumlah_kejadian',
    # Log-transform
    'log_korban_jiwa', 'log_people_affected', 'log_material_damage',
    # Flag
    'ada_korban_jiwa', 'ada_pengungsi', 'ada_material', 'ada_agri',
    # Kelengkapan laporan dampak
    'core_impact_missing_count', 'has_full_impact_report',
]
clf_post_features = [c for c in clf_post_features if c in df.columns]

# Skenario B: Pre-event (hanya info kontekstual, untuk prediksi potensi)
clf_pre_features = [
    'kejadian_code', 'provinsi_code', 'kabkot_code',
    'year', 'month', 'quarter', 'musim_hujan',
    'month_sin', 'month_cos',
    'kejadian_mean_severity', 'provinsi_mean_severity',
    'kabkot_total_laporan', 'kabkot_type_freq',
]
clf_pre_features = [c for c in clf_pre_features if c in df.columns]

# ── Simpan dataset klasifikasi ────────────────────────────────────────────────
# Baris berlabel 'Tidak Diketahui' (severity_code NaN) TIDAK dipakai training.
target_cols = ['severity', 'severity_code', 'severity_score']
clf_all_cols = list(dict.fromkeys(clf_post_features + target_cols))
clf_all_cols = [c for c in clf_all_cols if c in df.columns]

df_clf = df[clf_all_cols].copy()
n_before = len(df_clf)
df_clf = df_clf.dropna(subset=['severity_code'])
n_dropped = n_before - len(df_clf)

clf_path = OUT / 'bnpb_disaster_clf_ready.csv'
df_clf.to_csv(clf_path, index=False, encoding='utf-8-sig')
print(f'✓ Dataset klasifikasi disimpan: {clf_path}')
print(f'  Shape: {df_clf.shape}')
print(f'  Post-event features: {len(clf_post_features)}')
print(f'  Pre-event features : {len(clf_pre_features)}')
print(f'  Baris tanpa label severity (dampak tidak lengkap): {n_dropped:,} '
      f'({n_dropped / n_before * 100:.1f}%)')

# Distribusi target
print('\nDistribusi target severity_code:')
display(df_clf['severity'].value_counts().reindex(order).to_frame('count').assign(
    pct=lambda d: (d['count'] / len(df_clf) * 100).round(2)))

In [ ]:
# ── Cek class imbalance & rekomendasi ─────────────────────────────────────────
counts = df_clf['severity'].value_counts()
min_class = counts.min()
max_class = counts.max()
imbalance_ratio = max_class / min_class

print(f'Imbalance ratio (max/min): {imbalance_ratio:.1f}x')
if imbalance_ratio > 5:
    print('⚠ Imbalance signifikan → gunakan:')
    print('   - class_weight="balanced" di sklearn')
    print('   - SMOTE (imblearn) untuk oversampling minoritas')
    print('   - Evaluasi dengan F1-macro / F1-weighted, bukan accuracy')
else:
    print('✓ Imbalance relatif ringan → stratified split sudah cukup')

# Visualisasi imbalance
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors_sev = ['#d62728', '#ff7f0e', '#2ca02c', '#aec7e8']
counts.reindex(order).plot(kind='bar', ax=axes[0], color=colors_sev,
                            title='Distribusi Kelas Target', rot=0)
axes[0].set_xlabel('Severity')
counts.reindex(order).plot(kind='bar', ax=axes[1], color=colors_sev,
                            logy=True, title='Distribusi Kelas Target (log scale)', rot=0)
axes[1].set_xlabel('Severity')
plt.tight_layout(); plt.show()

## 11. Dataset Siap Analisis Tren (Time Series)

In [ ]:
# ── Agregasi bulanan untuk analisis tren ──────────────────────────────────────
monthly_trend = df.groupby(df.date.dt.to_period('M')).agg(
    total_laporan=('date', 'size'),
    total_korban=('korban_total', lambda x: x.sum(min_count=1)),
    total_terdampak=('people_affected', lambda x: x.sum(min_count=1)),
    total_material=('material_damage', lambda x: x.sum(min_count=1)),
    total_kerugian=('kerugian_juta', lambda x: x.sum(min_count=1)),
    avg_severity=('severity_score', 'mean'),
    n_berat=('severity', lambda x: (x == 'Berat').sum()),
    n_sedang=('severity', lambda x: (x == 'Sedang').sum()),
    n_ringan=('severity', lambda x: (x == 'Ringan').sum()),
    n_tanpa=('severity', lambda x: (x == 'Tanpa Dampak').sum()),
    n_tidak_diketahui=('severity', lambda x: (x == 'Tidak Diketahui').sum()),
).reset_index()
monthly_trend['date_ts'] = monthly_trend.date.dt.to_timestamp()
monthly_trend['year']    = monthly_trend.date_ts.dt.year
monthly_trend['month']   = monthly_trend.date_ts.dt.month
monthly_trend['musim_hujan'] = monthly_trend.month.isin([11, 12, 1, 2, 3]).astype(int)

# Rolling features untuk forecasting
for w in [3, 6, 12]:
    monthly_trend[f'laporan_ma{w}'] = monthly_trend['total_laporan'].rolling(w, min_periods=1).mean()
    monthly_trend[f'laporan_lag{w}'] = monthly_trend['total_laporan'].shift(w)

monthly_trend['laporan_yoy'] = monthly_trend['total_laporan'].pct_change(12) * 100

trend_path = OUT / 'bnpb_disaster_monthly_trend.csv'
monthly_trend.to_csv(trend_path, index=False, encoding='utf-8-sig')
print(f'✓ Dataset tren bulanan disimpan: {trend_path}')
print(f'  Shape: {monthly_trend.shape}')
display(monthly_trend.tail(5))

In [ ]:
# ── Agregasi per jenis bencana × bulan (untuk per-type trend) ─────────────────
type_monthly = (df.groupby([df.date.dt.to_period('M'), 'kejadian'])
                .agg(laporan=('date', 'size'),
                     korban=('korban_total', lambda x: x.sum(min_count=1)),
                     terdampak=('people_affected', lambda x: x.sum(min_count=1)))
                .reset_index())
type_monthly['date_ts'] = type_monthly.date.dt.to_timestamp()

type_trend_path = OUT / 'bnpb_disaster_type_monthly.csv'
type_monthly.to_csv(type_trend_path, index=False, encoding='utf-8-sig')
print(f'✓ Tren per jenis bencana disimpan: {type_trend_path}  |  shape: {type_monthly.shape}')

## 12. Simpan Dataset Fitur Lengkap & Laporan

In [ ]:
# ── Kolom yang disimpan ke dataset fitur lengkap ──────────────────────────────
keep_cols = [
    # Identitas
    'date', 'year', 'month', 'quarter', 'dayofweek', 'week_of_year', 'year_month',
    'tahun', 'bulan', 'tanggal',
    'id_prov', 'id_kab', 'provinsi', 'kabkot', 'desa', 'lokasi',
    'id_jenis_bencana', 'kejadian',
    # Musim
    'musim_hujan', 'musim_fase', 'month_sin', 'month_cos', 'dow_sin', 'dow_cos',
    # Dampak mentah (NaN = tidak dilaporkan)
    'meninggal', 'hilang', 'terluka', 'menderita', 'mengungsi',
    'rumah_rusak', 'rumah_rusak_berat', 'rumah_rusak_sedang', 'rumah_rusak_ringan',
    'terendam', 'lahan', 'sawah', 'kebun', 'hutan',
    'jembatan_rusak', 'jalan', 'kerugian_juta', 'jumlah_kejadian',
    # Dampak turunan
    'korban_jiwa', 'korban_total', 'people_affected', 'human_impact_total',
    'material_damage', 'agri_damage', 'infra_damage', 'fasilitas_damage',
    # Log-transform
    'log_korban_jiwa', 'log_korban_total', 'log_people_affected',
    'log_material_damage', 'log_agri_damage', 'log_kerugian_juta',
    # Flag dampak
    'ada_korban_jiwa', 'ada_korban_total', 'ada_pengungsi', 'ada_menderita',
    'ada_material', 'ada_agri', 'ada_infra',
    'has_coordinates', 'has_kronologis',
    # Kelengkapan laporan dampak
    'impact_missing_count', 'impact_report_ratio', 'core_impact_missing_count',
    'has_impact_report', 'has_full_impact_report',
    # Severity
    'severity', 'severity_code', 'severity_score',
    # Encoding
    'kejadian_code', 'provinsi_code', 'kabkot_code',
    'kejadian_mean_severity', 'provinsi_mean_severity',
    # Historis
    'kabkot_total_laporan', 'kabkot_type_freq',
    # Spasial
    'lon', 'lat', 'region',
    # Musim OHE
]
# Tambah OHE musim & flag missing per kolom dampak
ohe_musim_cols = [c for c in df.columns if c.startswith('musim_') and c not in keep_cols]
keep_cols.extend(ohe_musim_cols)
keep_cols.extend([c for c in df.columns if c.startswith('flag_') and c.endswith('_missing')])
keep_cols = [c for c in keep_cols if c in df.columns]
keep_cols = list(dict.fromkeys(keep_cols))  # hapus duplikat jaga urutan

df_features = df[keep_cols].copy()
features_path = OUT / 'bnpb_disaster_features.csv'
df_features.to_csv(features_path, index=False, encoding='utf-8-sig')
print(f'✓ Dataset fitur lengkap disimpan: {features_path}')
print(f'  Shape: {df_features.shape}')
print(f"  Kolom flag missing: {len([c for c in df_features.columns if c.startswith('flag_')])}")

In [ ]:
# ── Laporan preprocessing ─────────────────────────────────────────────────────
report_log.append(f'Shape akhir (features): {df_features.shape}')
report_log.append(f'Shape akhir (cluster)  : {cluster_ready.shape}')
report_log.append(f'Shape akhir (clf)      : {df_clf.shape}')
report_log.append(f'Shape akhir (tren)     : {monthly_trend.shape}')

severity_full = df['severity'].value_counts().reindex(severity_order)
severity_full_pct = (severity_full / len(df) * 100).round(2)

report_text = '\n'.join([
    '=== PREPROCESSING REPORT DisasterWatch ===',
    f'Input  : {INPUT.name}',
    f'Input shape: {df_raw.shape}',
    '',
    '--- LOG PERUBAHAN ---',
    *report_log,
    '',
    '--- OUTPUT FILES ---',
    f'bnpb_disaster_clean.csv        : {df.shape}',
    f'bnpb_disaster_features.csv     : {df_features.shape}',
    f'bnpb_disaster_cluster_ready.csv: {cluster_ready.shape}',
    f'bnpb_province_cluster_ready.csv: {prov_cluster.shape}',
    f'bnpb_disaster_clf_ready.csv    : {df_clf.shape}',
    f'bnpb_disaster_monthly_trend.csv: {monthly_trend.shape}',
    f'bnpb_disaster_type_monthly.csv : {type_monthly.shape}',
    '',
    '--- KELENGKAPAN DAMPAK ---',
    f'Baris tanpa laporan dampak (semua core NaN): {(df.has_impact_report == 0).sum():,}',
    f'Baris dengan laporan dampak lengkap        : {(df.has_full_impact_report == 1).sum():,}',
    f'Rata-rata kolom dampak hilang per baris    : {df.impact_missing_count.mean():.1f} / {len(impact_cols)}',
    f'Baris tanpa label severity (Tidak Diketahui): {(df.severity == "Tidak Diketahui").sum():,}',
    '',
    '--- FITUR KLASIFIKASI ---',
    f'Post-event features ({len(clf_post_features)}): ' + ', '.join(clf_post_features),
    f'Pre-event  features ({len(clf_pre_features)}): ' + ', '.join(clf_pre_features),
    '',
    '--- DISTRIBUSI SEVERITY (seluruh baris) ---',
    pd.DataFrame({'count': severity_full, 'pct': severity_full_pct}).to_string(),
    '',
    '--- DISTRIBUSI SEVERITY (dataset klasifikasi; label diketahui) ---',
    df_clf['severity'].value_counts().to_string(),
])

report_path = OUT / 'preprocessing_report.txt'
report_path.write_text(report_text, encoding='utf-8')
print(report_text)

## 13. Validasi Akhir

In [ ]:
# ── Cek konsistensi semua output ──────────────────────────────────────────────
outputs = {
    'bnpb_disaster_features.csv'     : df_features,
    'bnpb_disaster_cluster_ready.csv': cluster_ready,
    'bnpb_disaster_clf_ready.csv'    : df_clf,
    'bnpb_disaster_monthly_trend.csv': monthly_trend,
}

print('=== Validasi Output Files ===')
all_ok = True
for name, df_out in outputs.items():
    path = OUT / name
    # Cek file exist
    exists = path.exists()
    # Cek tidak ada semua NaN di kolom numerik
    num_cols = df_out.select_dtypes(include='number').columns
    all_nan_cols = [c for c in num_cols if df_out[c].isna().all()]
    # Cek shape positif
    shape_ok = df_out.shape[0] > 0 and df_out.shape[1] > 0
    
    status = '✓' if (exists and shape_ok and len(all_nan_cols) == 0) else '✗'
    if status == '✗':
        all_ok = False
    print(f'{status} {name}: shape={df_out.shape}, exist={exists}, all_nan_cols={all_nan_cols}')

print()
if all_ok:
    print('✅ Semua output valid. Dataset siap untuk EDA lanjutan, Clustering, dan Klasifikasi.')
else:
    print('⚠ Ada output yang perlu dicek ulang.')

In [ ]:
# ── Ringkasan fitur akhir ─────────────────────────────────────────────────────
print('\n=== RINGKASAN FITUR DATASET FEATURES ===')
feature_groups = {
    'Identitas & Waktu' : [c for c in df_features.columns
                           if any(c.startswith(p) for p in ['year', 'month', 'quarter', 'day', 'week', 'date', 'tahun', 'bulan', 'tanggal'])],
    'Musiman'           : [c for c in df_features.columns if 'musim' in c or 'sin' in c or 'cos' in c],
    'Lokasi'            : [c for c in df_features.columns if c in ['provinsi','kabkot','desa','lokasi','lon','lat','region','id_prov','id_kab']],
    'Dampak Mentah'     : [c for c in df_features.columns if c in ['meninggal','hilang','terluka','menderita','mengungsi','rumah_rusak','terendam','lahan','sawah','kebun','hutan','jembatan_rusak','jalan','kerugian_juta','jumlah_kejadian']],
    'Dampak Turunan'    : [c for c in df_features.columns if c in ['korban_jiwa','korban_total','people_affected','human_impact_total','material_damage','agri_damage','infra_damage','fasilitas_damage']],
    'Log-Transform'     : [c for c in df_features.columns if c.startswith('log_')],
    'Flag Biner'        : [c for c in df_features.columns if c.startswith('ada_') or c.startswith('has_')],
    'Severity Target'   : [c for c in df_features.columns if 'severity' in c],
    'Encoding'          : [c for c in df_features.columns if c.endswith('_code') or c.endswith('_mean_severity')],
    'Historis'          : [c for c in df_features.columns if 'freq' in c or 'laporan' in c],
}
for grp, cols in feature_groups.items():
    print(f'  {grp:20s}: {len(cols):3d} kolom  {cols[:5]} ...')

---
## Ringkasan Pipeline Preprocessing

```
RAW CSV  (dipilih: rentang tanggal terluas — sama dengan notebook EDA)
  │
  ├─ 1. Audit Kualitas (missing, duplikat, dtype)
  ├─ 2. Cleaning
  │     ├─ Konversi tipe data (numerik, string)
  │     ├─ Parsing tanggal & drop invalid
  │     ├─ Drop mandatory null & duplikat penuh
  │     └─ Clip anomali koordinat & nilai negatif
  │
  ├─ 3. Penanganan Missing Dampak
  │     ├─ Nilai dampak TETAP NaN (tidak diimputasi 0)
  │     ├─ flag_<kolom>_missing per kolom dampak
  │     └─ impact_missing_count / has_impact_report / has_full_impact_report
  │
  ├─ 4. Feature Engineering
  │     ├─ Temporal: year/month/quarter/dayofweek/musim/sin-cos
  │     ├─ Dampak: korban_total, people_affected, severity_score (NaN-aware)
  │     ├─ Severity label (4 kelas + 'Tidak Diketahui')
  │     ├─ Spasial: region, koordinat valid
  │     └─ Historis: frekuensi kabkot × jenis
  │
  ├─ 5. Encoding: Label + Target + OHE
  ├─ 6. Scaling: RobustScaler + MinMaxScaler (NaN dipertahankan)
  │
  └─ OUTPUT
        ├─ bnpb_disaster_clean.csv          ← bersih, tanpa engineering
        ├─ bnpb_disaster_features.csv       ← semua fitur lengkap
        ├─ bnpb_disaster_cluster_ready.csv  ← siap KMeans/DBSCAN (kabkot)
        ├─ bnpb_province_cluster_ready.csv  ← siap KMeans/DBSCAN (provinsi)
        ├─ bnpb_disaster_clf_ready.csv      ← siap klasifikasi severity
        ├─ bnpb_disaster_monthly_trend.csv  ← siap time series / forecasting
        ├─ bnpb_disaster_type_monthly.csv   ← tren per jenis bencana
        └─ preprocessing_report.txt         ← log perubahan
```

**Catatan missing value:** NaN berarti "tidak dilaporkan", bukan nol. Baris dengan
komponen penentu severity yang tidak lengkap diberi label `Tidak Diketahui`
(`severity_code = NaN`) dan otomatis dikeluarkan dari dataset klasifikasi.